# 실습 5 — 준비. `openai` SDK → LangChain · Streamlit 골격 (Day 3 / 준비 10~15분)

> 시나리오: **실습 4의 챗봇을 웹으로 올리기 직전** — 두 가지 다리를 먼저 건넌다.
>
> 1. 실습 3·4 는 `openai` SDK 를 직접 썼다. 웹 챗봇(`web/`)은 **LangChain** 으로 같은 일을 한다.
>    → *무엇이 1:1 로 대응되는지* 먼저 눈으로 확인한다 (LangChain 정식 강의는 실습 5 뒤에 나온다).
> 2. Streamlit 의 `session_state` 멘탈모델을 잡은 뒤, 완성본 `web/streamlit_app.py` 를 연다.

## 흐름
**openai 챗봇(복습) → 똑같은 호출을 LangChain 으로 → 스트리밍 1:1 → history 변환 → Streamlit 골격 → 완성본 실행**

## 0) 준비 — `.env` + 두 SDK 임포트

In [11]:
from dotenv import load_dotenv
from openai import OpenAI                                   # 실습 3·4 방식
from langchain_openai import ChatOpenAI                     # 실습 5(웹) 방식
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()
MODEL = "gpt-4o-mini"
SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."
print("ready:", MODEL)

ready: gpt-4o-mini


## 1) 복습 — `openai` SDK 로 한 번 호출 (실습 3·4 와 동일)

In [12]:
client = OpenAI()
resp = client.chat.completions.create(
    model=MODEL,
    temperature=0.3,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "한 문장으로 자기소개 해줘."},
    ],
)
print(resp.choices[0].message.content)

안녕하세요, 저는 여러분의 질문에 답하고 도움을 드리기 위해 여기 있는 친절한 AI 어시스턴트입니다.


## 2) 똑같은 호출을 **LangChain** 으로 — 1:1 대응

`dict` 메시지가 **클래스**로, `client.create(...)` 가 **`llm.invoke(...)`** 로 바뀔 뿐 개념은 같다.

| 개념 | `openai` SDK (실습 3·4) | LangChain (실습 5·6) |
|---|---|---|
| 클라이언트 | `OpenAI()` | `ChatOpenAI(model=, temperature=)` |
| system 메시지 | `{"role":"system", "content":...}` | `SystemMessage(content=...)` |
| 사용자 메시지 | `{"role":"user", "content":...}` | `HumanMessage(content=...)` |
| 모델 답변 | `{"role":"assistant", "content":...}` | `AIMessage(content=...)` |
| 호출 | `client.chat.completions.create(messages=...)` | `llm.invoke(msgs)` |
| 응답 텍스트 | `resp.choices[0].message.content` | `resp.content` |

In [13]:
llm = ChatOpenAI(model=MODEL, temperature=0.3)

msgs = [
    SystemMessage(content=SYSTEM),
    HumanMessage(content="한 문장으로 자기소개 해줘."),
]
resp = llm.invoke(msgs)
print(resp.content)        # ← openai 의 resp.choices[0].message.content 와 같은 자리

안녕하세요, 저는 여러분의 질문에 답하고 도움을 드리기 위해 여기 있는 AI 어시스턴트입니다.


## 3) 스트리밍도 1:1

실습 3 의 `chunk.choices[0].delta.content` 가 LangChain 에선 `chunk.content` 가 된다.
웹 챗봇이 글자를 또르륵 흘리는 바로 그 코드다.

In [14]:
for chunk in llm.stream(msgs):       # openai: for chunk in stream: chunk.choices[0].delta.content
    print(chunk.content or "", end="")
print()

안녕하세요, 저는 여러분의 질문에 답하고 도움을 드리기 위해 여기 있는 친절한 AI 어시스턴트입니다.


## 4) 핵심 다리 — 누적 history(dict) → LangChain 메시지

웹 챗봇은 화면 메시지를 `{"role", "content"}` **dict 리스트**로 보관하고(=실습 4 의 `history`),
호출 직전에 LangChain 메시지로 변환한다. `web/streamlit_app.py` 와 **완전히 같은 변환 루프**다.

In [15]:
# 화면에 쌓인 대화 (실습 4 의 history 와 동일한 구조)
messages = [
    {"role": "user", "content": "내 이름은 박성용이야."},
    {"role": "assistant", "content": "네, 박성용님 반가워요!"},
    {"role": "user", "content": "방금 내 이름이 뭐였지?"},
]

# system + 누적 history 를 LangChain 메시지로 변환
msgs = [SystemMessage(content=SYSTEM)]
for m in messages:
    cls = HumanMessage if m["role"] == "user" else AIMessage
    msgs.append(cls(content=m["content"]))

print(llm.invoke(msgs).content)      # 앞 턴을 기억하는지 확인

당신의 이름은 박성용입니다.


In [ ]:
# 입출력 예시 셀 (새로 추가)
# 입력: messages 리스트 (role/content 딕셔너리)
messages_input = [
    {"role": "user", "content": "내 이름은 박성용이야."},
    {"role": "assistant", "content": "네, 박성용님 반가워요!"},
    {"role": "user", "content": "방금 내 이름이 뭐였지?"},
]

# system + 누적 history 를 LangChain 메시지로 변환 (입력 → 출력)
msgs = [SystemMessage(content=SYSTEM)]
for m in messages_input:
    cls = HumanMessage if m["role"] == "user" else AIMessage
    msgs.append(cls(content=m["content"]))

print('\n--- 입력(messages_input) ---')
for m in messages_input:
    print(m)
print('\n--- 출력(llm.invoke(msgs).content) ---')
print(llm.invoke(msgs).content)

## 5) Streamlit 멘탈모델 — `session_state` 3가지만 기억

Streamlit 은 **사용자가 입력할 때마다 스크립트를 처음부터 다시 실행**(rerun)한다.
그래서 "기억"이 필요한 값은 보통 변수가 아니라 `st.session_state` 에 둔다.

| 위젯/객체 | 하는 일 |
|---|---|
| `st.session_state` | rerun 에도 **살아남는 변수 보관함**. 대화 기록을 여기 둔다. |
| `st.chat_message(role).write(text)` | 말풍선 1개 렌더 (`user` / `assistant`) |
| `st.chat_input(...)` | 하단 입력창. 입력이 들어오면 그 값을 반환하며 **rerun** |
| `st.write_stream(gen)` | 제너레이터를 받아 글자를 **또르륵 스트리밍** + 최종 문자열 반환 |

아래는 노트북이 아니라 `.py` 로 `streamlit run` 해야 도는 **최소 골격**이다 (실행하지 말고 구조만 본다):

```python
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

if "messages" not in st.session_state:        # rerun 에도 살아남는 대화 기록
    st.session_state.messages = []

for m in st.session_state.messages:           # 지난 대화 다시 그리기
    st.chat_message(m["role"]).write(m["content"])

if user_input := st.chat_input("메시지"):      # 입력 → 여기서 rerun
    st.session_state.messages.append({"role": "user", "content": user_input})
    st.chat_message("user").write(user_input)

    msgs = [SystemMessage(content=SYSTEM)]     # ← 4) 에서 본 변환 루프 그대로
    for m in st.session_state.messages:
        cls = HumanMessage if m["role"] == "user" else AIMessage
        msgs.append(cls(content=m["content"]))

    with st.chat_message("assistant"):
        answer = st.write_stream(c.content or "" for c in llm.stream(msgs))
    st.session_state.messages.append({"role": "assistant", "content": answer})
```

## 6) 완성본 실행 → 변형(TODO)

위 골격에 **페르소나 선택·temperature 슬라이더·환영 메시지·FAQ·누적 비용·오류 가드**를 더한 것이
`web/streamlit_app.py` 다. 컨테이너 터미널에서:

```bash
streamlit run web/streamlit_app.py     # 8501 자동 포워딩
```

### 직접 해보기 (TODO)
- [ ] `web/streamlit_app.py` 를 열어 **4) 의 변환 루프**가 어디 있는지 찾아 표시한다.
- [ ] 사이드바에서 페르소나·temperature 를 바꿔 말투/일관성 차이를 확인한다.
- [ ] `PERSONAS` 에 나만의 페르소나 1개를 추가한다.
- [ ] (비교) `python web/gradio_app.py` 로 **같은 챗봇을 Gradio** 로 띄워 본다 — 내부 변환 루프는 동일하다.

### 회고
- [ ] `openai` SDK 와 LangChain 메시지의 1:1 대응을 한 줄로 설명할 수 있는가?
- [ ] `session_state` 가 없으면 대화가 왜 사라지는가? (rerun)